In [ ]:
from pathlib import Path

import pandas as pd

from application.main import normalize_sections
from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.4")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)

In [ ]:
df_base_registry, df_extraction_registry = normalize_sections(df_base_registry, 
                                                              df_extraction_registry,
                                                              version_object,
                                                              section_type=["CAS"])

In [ ]:
from application.pipeline import data_cleaner_single_file

path = Path("data/records/software_version/v1.0.3/pipeline_version/v1.0.0/10.1007_JHEP01(2025)045/CAS_section.txt")

doc_doi = "10.1007/JHEP01(2025)045"

folder_doi = "10.1007_JHEP01(2025)045"

list = []

dependency_set = set(list)

df_base_registry, df_extraction_registry, dependency_sha = data_cleaner_single_file(path, doc_doi, folder_doi, df_base_registry, df_extraction_registry, version_object, dependency_set)


In [ ]:
version_101 = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.1")
version_102 = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.2")

df_base_registry, df_extraction_registry = normalize_sections(
    df_base_registry, df_extraction_registry, version_101, force_processing=True
)

df_base_registry, df_extraction_registry = normalize_sections(
    df_base_registry, df_extraction_registry, version_102, force_processing=True
)

In [ ]:
df_base_registry, df_extraction_registry = normalize_sections(df_base_registry, df_extraction_registry, version_object)

In [ ]:
print(df_extraction_registry.duplicated(subset=["output_sha"]).sum())

In [ ]:
merged = df_extraction_registry.merge(df_base_registry[["output_sha", "software_version"]], on="output_sha", how="left")
print(merged.groupby(["stage", "software_version"]).size())

In [ ]:
from application.utils import save_registry

# 1. deduplicate extraction registry keeping last (most recent)
df_extraction_registry = df_extraction_registry.drop_duplicates(subset=["output_sha"], keep="last")

# 2. deduplicate base registry
df_base_registry = df_base_registry.drop_duplicates(subset=["output_sha"], keep="last")

# 3. check what's left
print(df_extraction_registry.groupby("stage").size())
print(df_base_registry.groupby("software_version").size())

# 4. save clean versions
save_registry(df_base_registry, BASE_REGISTRY)
save_registry(df_extraction_registry, EXTRACTION_REGISTRY)

In [ ]:
merged = df_extraction_registry.merge(df_base_registry[["output_sha", "software_version"]], on="output_sha", how="left")
print(merged.groupby(["stage", "software_version"]).size())

In [ ]:
# check distribution by software version
print(df_extraction_registry.groupby("stage").size())
print(df_base_registry.groupby("software_version").size())

In [ ]:
# paths referenced in the code below use absolute paths relative to the original local environment

from application.pipeline import data_cleaner_single_file
from application.utils import save_registry

dependency_set = set(
    df_base_registry.loc[
        df_base_registry["output_sha"].isin(
            df_extraction_registry.loc[df_extraction_registry["stage"] == "cleaned", "output_sha"]
        ),
        "dependencies",
    ].dropna()
)
DAS_path_1 = Path("data/records/software_version/v1.0.2/pipeline_version/v1.0.0/10.1103_95q6-vvlp/DAS_section.txt")
doc_doi_1 = "10.1103/95q6-vvlp"
folder_doi_1 = "10.1103_95q6-vvlp"
df_base_registry, df_extraction_registry, dependency_sha = data_cleaner_single_file(DAS_path_1, doc_doi_1, folder_doi_1, df_base_registry, df_extraction_registry, version_object, dependency_set, force_processing=True)
DAS_path_2 = Path("data/records/software_version/v1.0.2/pipeline_version/v1.0.0/10.1103_PhysRevD.111.092014/DAS_section.txt")
doc_doi_2 = "10.1103/PhysRevD.111.092014"
folder_doi_2 = "10.1103_PhysRevD.111.092014"
df_base_registry, df_extraction_registry, dependency_sha = data_cleaner_single_file(DAS_path_2, doc_doi_2, folder_doi_2, df_base_registry, df_extraction_registry, version_object, dependency_set, force_processing=True)
DAS_path_3 = Path("data/records/software_version/v1.0.2/pipeline_version/v1.0.0/10.1103_PhysRevD.111.112004/DAS_section.txt")
doc_doi_3 = "10.1103/PhysRevD.111.112004"
folder_doi_3 = "10.1103_PhysRevD.111.112004"
df_base_registry, df_extraction_registry, dependency_sha = data_cleaner_single_file(DAS_path_3, doc_doi_3, folder_doi_3, df_base_registry, df_extraction_registry, version_object, dependency_set, force_processing=True)
save_registry(df_base_registry, BASE_REGISTRY)
save_registry(df_extraction_registry, EXTRACTION_REGISTRY)

In [ ]:
df_extraction_registry.duplicated().sum()

In [ ]:
df_base_registry = pd.read_csv("data/metadata/registries/base_output_registry.csv")
merge_base_extraction = df_extraction_registry.merge(df_base_registry[["doc_doi", "output_sha", "software_version"]], on="output_sha", how="left")
df_cleaned_texts = merge_base_extraction.loc[(merge_base_extraction["stage"]=="cleaned")&(merge_base_extraction["software_version"]=="v1.0.2"),["text", "doc_doi"]]
display(df_cleaned_texts.value_counts().reset_index(name="count"))